# Photodetection Statistical Fluctuation

October/2026  
NOMACHI, Masaharu

An activity of photon-counting detection. This version uses Bokeh for plotting.

In [26]:
# configuration file
import numpy as np
import pandas as pd
import cairosvg



from bokeh.io import output_notebook, show as _show, export_svgs, export_png
from bokeh.models import (
    Label, Span, HoverTool, ColumnDataSource, CrosshairTool,
    Div, CustomJS
)
from bokeh.plotting import figure as _figure
from bokeh.layouts import column

# Figura personalizada con coordenadas del cursor
def figure(*args, **kwargs):
    kwargs.setdefault("width", 1000)
    kwargs.setdefault("height", 500)
    kwargs.setdefault(
        "tools",
        "pan,wheel_zoom,box_zoom,reset,save"
    )

    p = _figure(*args, **kwargs)

    coords = Div(
        text="Coordenadas: mueve el cursor sobre la gráfica"
    )

    p.js_on_event(
        "mousemove",
        CustomJS(
            args=dict(coords=coords),
            code="""
                coords.text =
                    `x = ${cb_obj.x.toFixed(4)}, ` +
                    `y = ${cb_obj.y.toFixed(4)}`;
            """
        )
    )

    p._coords_div = coords
    return p


# Mostrar figuras con las coordenadas automáticamente
def show(obj, *args, **kwargs):
    if hasattr(obj, "_coords_div"):
        obj = column(obj, obj._coords_div)

    return _show(obj, *args, **kwargs)

import tempfile
import os

def save_pdf(fig, filepath):
    fig.output_backend = "svg"

    with tempfile.NamedTemporaryFile(suffix=".svg") as tmp:
        export_svgs(fig, filename=tmp.name)
        cairosvg.svg2pdf(url=tmp.name, write_to=filepath)

def save_png(fig, filepath, size=500):
    fig.width = size
    fig.height = None
    fig.aspect_ratio = 2  # width:height = 2:1
    fig.toolbar_location = None
    fig.toolbar.tools = []
    export_png(fig, filename=filepath)

output_notebook()

Loading BokehJS ...

We observed the peaks referred to as *photon counting*.

In [ ]:
# Data import and visualization
data = [
    pd.read_csv(file, usecols=["Center_V", "Counts"]).rename(
        columns={"Center_V": "voltage", "Counts": "counts"}
    )
    for file in ["MPPC_1.csv", "MPPC_2.csv"]
]

bar_width = data[1]["voltage"].diff().abs().iloc[1]

photon_histogram_1 = figure(
    title="Photon counting",
    x_axis_label="Voltage (V)",
    y_axis_label="Counts",
)

photon_histogram_1.vbar(
    x="voltage",
    top="counts",
    width=data[1]["voltage"].diff().abs().iloc[1],
    fill_color="#1f77b4",
    line_color="white",
    source=ColumnDataSource(data[1]),
)

photon_histogram_1.add_tools(HoverTool(
    tooltips=[
        ("Voltage", "@voltage{0.000} V"),
        ("Counts", "@counts"),
    ],
    mode="mouse",
))

show(photon_histogram_1)

save_png(photon_histogram_1, "figs/photon_counting.png")

## Data analysis

Since the first data set has only one peak, we use voltage windows to identify the photon-number peaks.

### Windows - Data 1

In [31]:
windows_rough = np.array([0.06, 0.2, 0.34, 0.46, 0.60, 0.72])

# Locate the measured voltage closest to each approximate separator.
indexes = [(data[1]["voltage"] - target).abs().idxmin() for target in windows_rough]
windows = data[1].loc[indexes, ["voltage"]].copy()

window_histogram = figure(
    title="Photon counting with separation windows",
    x_axis_label="Voltage (V)",
    y_axis_label="Counts",
)
window_histogram.vbar(  # histogram bars
    x="voltage",
    top="counts",
    width=bar_width,
    fill_color="#1f77b4",
    line_color="white",
    source=ColumnDataSource(data[1]),
)

# Límites del eje y
y_max = float(data[1]["counts"].max())
window_histogram.y_range.start = 0
window_histogram.y_range.end = y_max * 1.25

# Líneas divisorias y etiquetas
for edge in windows["voltage"]:
    right_edge = float(edge) + bar_width / 2.0

    window_histogram.add_layout(
        Span(
            location=right_edge,
            dimension="height",
            line_color="red",
            line_width=1,
        )
    )

    window_histogram.add_layout(
        Label(
            x=right_edge,
            y=y_max * 1.08,
            x_units="data",
            y_units="data",
            text=f"{edge:.2f} V",
            text_font_size="10pt",
            text_color="red",
            text_align="center",
            text_baseline="bottom",
            background_fill_color="white",
            background_fill_alpha=0.9,
            level="overlay",
        )
    )

window_histogram.add_tools(
    HoverTool(
        tooltips=[
            ("Voltage", "@voltage{0.000} V"),
            ("Counts", "@counts"),
        ],
        mode="mouse",
    ),
    CrosshairTool(line_color="gray", line_alpha=0.3)
)

show(window_histogram)

save_png(window_histogram, "figs/window_histogram_1.png")

In [13]:
# Summarize counts in each window
voltage = data[1]["voltage"]
counts = data[1]["counts"]
edges = np.r_[-np.inf, windows["voltage"].to_numpy(), np.inf]

window_index = pd.cut(voltage, bins=edges, right=True, labels=False)
stats = (
    pd.DataFrame({"photon_number": window_index, "counts": counts})
    .groupby("photon_number", observed=False)["counts"]
    .sum()
    .reset_index(name="sum_counts")
)
stats["limits"] = [[float(a), float(b)] for a, b in zip(edges[:-1], edges[1:])]
stats = stats[["limits", "photon_number", "sum_counts"]]

weighted_average = np.average(stats["photon_number"], weights=stats["sum_counts"])
variance = np.average(
    (stats["photon_number"] - weighted_average) ** 2, weights=stats["sum_counts"]
)

print("Weighted average photon number:", weighted_average)
print("Variance of photon number:", variance)
stats

Weighted average photon number: 4.262023217247098
Variance of photon number: 2.156882805431109


,limits,photon_number,sum_counts
0,"[-inf, 0.0625]",0,5
1,"[0.0625, 0.1975]",1,22
2,"[0.1975, 0.3415]",2,56
3,"[0.3415, 0.4585]",3,87
4,"[0.4585, 0.6025]",4,145
5,"[0.6025, 0.7195]",5,133
6,"[0.7195, inf]",6,155


The next plot shows the photon distribution and a Poisson fit.

$$\lambda = \langle n \rangle = \sigma^2$$

$$P_n = \frac{\lambda^n}{n!}e^{-\lambda}$$

In [27]:
# Fit a Poisson distribution to the photon counting data
from scipy.optimize import curve_fit
from scipy.stats import poisson

# fit--------------------------------
photon_number = stats["photon_number"].to_numpy(dtype=int)
sum_counts = stats["sum_counts"].to_numpy(dtype=float)

def poisson_counts(k, mu):
    return sum_counts.sum() * poisson.pmf(k, mu)

params, covariance = curve_fit(
    poisson_counts, photon_number, sum_counts, p0=[weighted_average], bounds=(1e-9, np.inf)
)
fitted_mu = params[0]
mu_std = np.sqrt(covariance[0, 0])

# plot---------------------------------

distribution = figure(
    title="Photon distribution with Poisson fit",
    x_axis_label="Photon number", y_axis_label="Counts",
)
distribution.scatter(
    photon_number, sum_counts, size=12, color="darkred", legend_label="Measured counts"
)
distribution.add_tools(
    HoverTool(
        tooltips=[
            ("Photon number", "$x{0}"),
            ("Counts", "$y{0}"),
        ],
        mode="mouse",
    ),
)

distribution.line(
    photon_number, poisson_counts(photon_number, fitted_mu), line_width=3, color="royalblue",
    legend_label=f"Poisson fit (mu={fitted_mu:.2f} +/- {mu_std:.2f})",
)
distribution.scatter(
    photon_number, poisson_counts(photon_number, fitted_mu), size=7, color="royalblue"
)
distribution.add_layout(Label(
    x=10, y=470, x_units="screen", y_units="screen",
    text=(f"Weighted average = {weighted_average:.2f}\n"
          f"Variance = {variance:.2f}\n"
          f"Fitted mu = {fitted_mu:.2f}"),
    text_font_size="10pt", background_fill_color="white",
    background_fill_alpha=0.85, border_line_color="black"
))
distribution.legend.location = "top_left"


show(distribution)

save_png(distribution, "figs/poisson_fit_1.png")

**Conclusions**
There is a good fit for the first 4 points, next we cannot assure reliability due to saturation of last point.

### Windows - Data 0

In [28]:
windows_rough = np.array([0.06, 0.2, 0.34, 0.46, 0.60, 0.72])

# Locate the measured voltage closest to each approximate separator.
indexes = [(data[0]["voltage"] - target).abs().idxmin() for target in windows_rough]
windows = data[0].loc[indexes, ["voltage"]].copy()

window_histogram = figure(
    title="Photon counting with separation windows",
    x_axis_label="Voltage (V)",
    y_axis_label="Counts",
)
window_histogram.vbar(  # histogram bars
    x="voltage",
    top="counts",
    width=bar_width,
    fill_color="#1f77b4",
    line_color="white",
    source=ColumnDataSource(data[0]),
)

# Límites del eje y
y_max = float(data[0]["counts"].max())
window_histogram.y_range.start = 0
window_histogram.y_range.end = y_max * 1.25

# Líneas divisorias y etiquetas
for edge in windows["voltage"]:
    right_edge = float(edge) + bar_width / 2.0

    window_histogram.add_layout(
        Span(
            location=right_edge,
            dimension="height",
            line_color="red",
            line_width=1,
        )
    )

    window_histogram.add_layout(
        Label(
            x=right_edge,
            y=y_max * 1.08,
            x_units="data",
            y_units="data",
            text=f"{edge:.2f} V",
            text_font_size="10pt",
            text_color="red",
            text_align="center",
            text_baseline="bottom",
            background_fill_color="white",
            background_fill_alpha=0.9,
            level="overlay",
        )
    )

window_histogram.add_tools(
    HoverTool(
        tooltips=[
            ("Voltage", "@voltage{0.000} V"),
            ("Counts", "@counts"),
        ],
        mode="mouse",
    ),
    CrosshairTool(line_color="gray", line_alpha=0.3)
)

# save_pdf(window_histogram, "figs/window_histogram.pdf")

show(window_histogram)

save_png(window_histogram, "figs/window_histogram_0.png")

In [29]:
# Summarize data[0] counts in each voltage window
data0_voltage = data[0]["voltage"]
data0_counts = data[0]["counts"]
data0_edges = np.r_[-np.inf, windows["voltage"].to_numpy(), np.inf]

data0_window_index = pd.cut(
    data0_voltage, bins=data0_edges, right=True, labels=False
)
data0_stats = (
    pd.DataFrame({"photon_number": data0_window_index, "counts": data0_counts})
    .groupby("photon_number", observed=False)["counts"]
    .sum()
    .reset_index(name="sum_counts")
)
data0_stats["limits"] = [
    [float(a), float(b)] for a, b in zip(data0_edges[:-1], data0_edges[1:])
]
data0_stats = data0_stats[["limits", "photon_number", "sum_counts"]]

data0_weighted_average = np.average(
    data0_stats["photon_number"], weights=data0_stats["sum_counts"]
)
data0_variance = np.average(
    (data0_stats["photon_number"] - data0_weighted_average) ** 2,
    weights=data0_stats["sum_counts"],
)

print("Data[0] weighted average photon number:", data0_weighted_average)
print("Data[0] variance of photon number:", data0_variance)
data0_stats

Data[0] weighted average photon number: 0.5488372093023256
Data[0] variance of photon number: 0.6048242293131423


,limits,photon_number,sum_counts
0,"[-inf, 0.0625]",0,1264
1,"[0.0625, 0.1975]",1,661
2,"[0.1975, 0.3415]",2,174
3,"[0.3415, 0.4585]",3,36
4,"[0.4585, 0.6025]",4,12
5,"[0.6025, 0.7195]",5,3
6,"[0.7195, inf]",6,0


In [30]:
# Fit a Poisson distribution to data[0]
data0_photon_number = data0_stats["photon_number"].to_numpy(dtype=int)
data0_sum_counts = data0_stats["sum_counts"].to_numpy(dtype=float)

def data0_poisson_counts(k, mu):
    return data0_sum_counts.sum() * poisson.pmf(k, mu)

data0_params, data0_covariance = curve_fit(
    data0_poisson_counts,
    data0_photon_number,
    data0_sum_counts,
    p0=[data0_weighted_average],
    bounds=(1e-9, np.inf),
)
data0_fitted_mu = data0_params[0]
data0_mu_std = np.sqrt(data0_covariance[0, 0])

print(f"Data[0] fitted mu: {data0_fitted_mu:.4f} +/- {data0_mu_std:.4f}")

data0_distribution = figure(
    title="Data[0] photon distribution with Poisson fit",
    x_axis_label="Photon number",
    y_axis_label="Counts",
)
data0_distribution.scatter(
    data0_photon_number,
    data0_sum_counts,
    size=12,
    color="darkred",
    legend_label="Measured counts",
)
data0_distribution.line(
    data0_photon_number,
    data0_poisson_counts(data0_photon_number, data0_fitted_mu),
    line_width=3,
    color="royalblue",
    legend_label=f"Poisson fit (mu={data0_fitted_mu:.2f} +/- {data0_mu_std:.2f})",
)
data0_distribution.legend.location = "top_right"
show(data0_distribution)

save_png(data0_distribution, "figs/poisson_fit_0.png")

Data[0] fitted mu: 0.5278 +/- 0.0038


**Conclusion** As every point fits well in the distribution, **we conclude Poisson distribution** is correct for predict our model. In this case we don't get any saturation which could affect last measurement.